In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SQLServer-JDBC-Test") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
from pyspark.sql import functions as F

test_df = spark.range(0, 10000).withColumn(
    "name",
    F.concat(F.lit("test_"), F.col("id"))
)

print("Rows:", test_df.count())

Rows: 10000


In [3]:
jdbc_url = "jdbc:sqlserver://localhost:9999;databaseName=TestDB"

test_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "dbo.test_table") \
    .option("user", "testuser") \
    .option("password", "testpassword") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .mode("append") \
    .save()

Py4JJavaError: An error occurred while calling o43.save.
: java.lang.ClassNotFoundException: com.microsoft.sqlserver.jdbc.SQLServerDriver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcOptionsInWrite.<init>(JDBCOptions.scala:246)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcOptionsInWrite.<init>(JDBCOpt

In [4]:
import glob

jars = glob.glob("/opt/spark/jars/*sqlserver*") + \
       glob.glob("/usr/local/spark/jars/*sqlserver*")

print(jars)

[]


In [5]:
import os

print("Spark home:", os.environ.get("SPARK_HOME"))
print("Java home:", os.environ.get("JAVA_HOME"))

Spark home: /usr/local/spark
Java home: None


In [9]:
spark.stop()

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SQLServer-JDBC-Test") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.jars", "/usr/local/spark/jars/mssql-jdbc-13.4.0.jre11.jar") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [13]:
print("SPARK_HOME:", spark.sparkContext.getConf().get("spark.home"))
print("spark.jars:", spark.sparkContext.getConf().get("spark.jars"))
print("classpath:", spark.sparkContext.getConf().get("spark.driver.extraClassPath"))

SPARK_HOME: None
spark.jars: /usr/local/spark/jars/mssql-jdbc-13.4.0.jre11.jar
classpath: None


In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--jars /usr/local/spark/jars/mssql-jdbc-13.4.0.jre11.jar "
    "pyspark-shell"
)

print("JDBC JAR configured")

JDBC JAR configured


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SQLServer-JDBC-Test") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [3]:
spark._jvm.java.lang.Class.forName(
    "com.microsoft.sqlserver.jdbc.SQLServerDriver"
)

print("SQL Server JDBC driver loaded successfully")

SQL Server JDBC driver loaded successfully


In [5]:
from pyspark.sql import functions as F

test_df = spark.range(0, 10000).withColumn(
    "name",
    F.concat(F.lit("test_"), F.col("id"))
)

print("Rows:", test_df.count())

Rows: 10000


In [6]:
jdbc_url = "jdbc:sqlserver://localhost:9999;databaseName=TestDB"

test_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "dbo.test_table") \
    .option("user", "testuser") \
    .option("password", "testpassword") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .mode("append") \
    .save()

Py4JJavaError: An error occurred while calling o46.save.
: com.microsoft.sqlserver.jdbc.SQLServerException: The TCP/IP connection to the host localhost, port 9999 has failed. Error: "Connection refused. Verify the connection properties. Make sure that an instance of SQL Server is running on the host and accepting TCP/IP connections at the port. Make sure that TCP connections to the port are not blocked by a firewall.".
	at com.microsoft.sqlserver.jdbc.SQLServerException.makeFromDriverError(SQLServerException.java:308)
	at com.microsoft.sqlserver.jdbc.SQLServerException.convertConnectExceptionToSQLServerException(SQLServerException.java:399)
	at com.microsoft.sqlserver.jdbc.SocketFinder.findSocket(IOBuffer.java:2618)
	at com.microsoft.sqlserver.jdbc.TDSChannel.open(IOBuffer.java:743)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectHelper(SQLServerConnection.java:4352)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.login(SQLServerConnection.java:3937)
	at com.microsoft.

In [8]:
jdbc_url = "jdbc:sqlserver://localhost:9998;databaseName=TestDB"

test_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "dbo.test_table") \
    .option("user", "testuser") \
    .option("password", "testpassword") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .mode("append") \
    .save()

Py4JJavaError: An error occurred while calling o64.save.
: com.microsoft.sqlserver.jdbc.SQLServerException: The TCP/IP connection to the host localhost, port 9998 has failed. Error: "Connection refused. Verify the connection properties. Make sure that an instance of SQL Server is running on the host and accepting TCP/IP connections at the port. Make sure that TCP connections to the port are not blocked by a firewall.".
	at com.microsoft.sqlserver.jdbc.SQLServerException.makeFromDriverError(SQLServerException.java:308)
	at com.microsoft.sqlserver.jdbc.SQLServerException.convertConnectExceptionToSQLServerException(SQLServerException.java:399)
	at com.microsoft.sqlserver.jdbc.SocketFinder.findSocket(IOBuffer.java:2618)
	at com.microsoft.sqlserver.jdbc.TDSChannel.open(IOBuffer.java:743)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectHelper(SQLServerConnection.java:4352)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.login(SQLServerConnection.java:3937)
	at com.microsoft.

In [9]:
jdbc_url = "jdbc:sqlserver://localhost:9997;databaseName=TestDB"

test_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "dbo.test_table") \
    .option("user", "testuser") \
    .option("password", "testpassword") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("loginTimeout", "5") \
    .mode("append") \
    .save()

Py4JJavaError: An error occurred while calling o74.save.
: com.microsoft.sqlserver.jdbc.SQLServerException: Connection reset ClientConnectionId:cc46de1d-0e12-443b-8950-2d87cea6f720
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.terminate(SQLServerConnection.java:4904)
	at com.microsoft.sqlserver.jdbc.TDSChannel.read(IOBuffer.java:2202)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.prelogin(SQLServerConnection.java:4578)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectHelper(SQLServerConnection.java:4377)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.login(SQLServerConnection.java:3937)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectInternal(SQLServerConnection.java:3700)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connect(SQLServerConnection.java:2385)
	at com.microsoft.sqlserver.jdbc.SQLServerDriver.connect(SQLServerDriver.java:1453)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnectio

In [10]:
jdbc_url = "jdbc:sqlserver://localhost:9997;databaseName=TestDB"

test_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "dbo.test_table") \
    .option("user", "testuser") \
    .option("password", "testpassword") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("loginTimeout", "5") \
    .mode("append") \
    .save()

Py4JJavaError: An error occurred while calling o85.save.
: com.microsoft.sqlserver.jdbc.SQLServerException: Connection reset ClientConnectionId:0e5f10e5-7996-416d-a437-297ac6f3d9c9
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.terminate(SQLServerConnection.java:4904)
	at com.microsoft.sqlserver.jdbc.TDSChannel.read(IOBuffer.java:2202)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.prelogin(SQLServerConnection.java:4578)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectHelper(SQLServerConnection.java:4377)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.login(SQLServerConnection.java:3937)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectInternal(SQLServerConnection.java:3700)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connect(SQLServerConnection.java:2385)
	at com.microsoft.sqlserver.jdbc.SQLServerDriver.connect(SQLServerDriver.java:1453)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnectio